# Member 4 — D-309 Ablation Studies & Hybrid DL-XGBoost

Read-only summary of the saved Member 4 artifacts under decision **D-309**.

All ablations reference Member 3's winning D-307 Embedding MLP (config d307_016).
All tuning decisions and epoch selections were performed exclusively on August 2013 data.
September–October 2013 was scored once across five random seeds (42–46).
November–December 2013 remained strictly sealed and was never accessed.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULT = ROOT / "reports/results/phase2_member4"
FIGURE = ROOT / "reports/figures/phase2_member4"
print("Member 4 results loaded from:", RESULT)


## 1. August 2013 Tuning and Selection

Candidate models were trained on January–July 2013 and evaluated on August 2013 to select stopping epochs and tree counts.


In [2]:
selection_df = pd.read_csv(RESULT / "august_seed_selection.csv")
display(selection_df)


## 2. September–October 2013 Evaluation & Ablations

Models refit on January–August 2013 evaluated on the frozen September–October 2013 test set (55,628 flights).


In [3]:
comparison_df = pd.read_csv(RESULT / "comparison.csv")
display(comparison_df[["model", "pr_auc_mean", "pr_auc_std", "roc_auc_mean", "brier_score_mean", "ece_10_mean"]])
display(Image(filename=str(FIGURE / "ablation_comparison.png")))
display(Image(filename=str(FIGURE / "hybrid_pr_curves.png")))


## 3. Paired Significance Testing (1,000 Day-Level Bootstrap Resamples)

95% confidence intervals for PR-AUC differences computed by block bootstrapping whole days.


In [4]:
bootstrap_df = pd.read_csv(RESULT / "bootstrap_pr_auc_differences.csv")
display(bootstrap_df)
display(Image(filename=str(FIGURE / "bootstrap_differences.png")))


## 4. Hybrid Architecture Diagram


In [5]:
display(Image(filename=str(FIGURE / "hybrid_architecture.png")))


## 5. Key Findings & Discussion

1. **Historical Features**: Removing the 29 historical features decreases PR-AUC from 0.2757 to 0.2576 (-0.0181 delta). Past delays and rolling performance are vital.
2. **Congestion Features**: Removing the 4 congestion features decreases PR-AUC to 0.2596 (-0.0161 delta, 95% CI [-0.0267, -0.0057]), confirming airport departure/arrival load is statistically significant.
3. **Learned Embeddings vs One-Hot**: Ablation 3 (one-hot encoding) achieves PR-AUC 0.2753 vs 0.2757 for embeddings (difference -0.0004, 95% CI [-0.0064, +0.0051], crosses zero). Embeddings compress dimension from 1,015 to 86 without loss of predictive discrimination.
4. **Regularization**: Disabling dropout and weight decay yielded PR-AUC 0.2757 (difference -0.00003, CI crosses zero), demonstrating early stopping is the primary regularizer.
5. **Hybrid DL + XGBoost**: The hybrid model (0.2643 PR-AUC) does not outperform either the pure Embedding MLP (0.2757) or plain XGBoost (0.2681), showing that tree ensembles on dense latent features struggle with co-adaptation compared to end-to-end backpropagation.
